# Section 1: Setup & Academic Context

## Problem 7: Siamese Network Sensitivity Analysis

### Research question

**What perturbations break similarity learning in Siamese-style embedding models?**  
We study a *zero-shot* regime: a frozen, pretrained convolutional backbone maps cropped road-obstacle patches to 1D embedding vectors. Pairs are formed from YOLO boxes in Indian road scenes. We ask how **rotation**, **blur**, and **occlusion**—at increasing severity—alter **cosine similarity** and **Euclidean ($L_2$) distance** between embeddings when one side of the pair is perturbed.

This isolates the **sensitivity of the representation** rather than a trained Siamese loss, but it matches the operational question: *when do shared-weight embeddings cease to preserve metric structure?*

### Theoretical framing & citations

- **Koch, Zemel, and Salakhutdinov (2015)** — *Siamese Neural Networks for One-shot Image Recognition.* Introduces the foundational **shared-weight Siamese architecture** for **distance-based / metric learning**, where two branches with tied parameters produce comparable embeddings for similarity decisions. We cite this to ground the **embedding + distance** view used in our sensitivity study.

- **Hendrycks & Dietterich (2019)** — *Benchmarking Neural Network Robustness to Common Corruptions and Perturbations (ImageNet-C).* Justifies **systematic corruption benchmarks** (including **blur** and **noise-like degradations**) as a standard way to characterize **robustness** of visual representations—not only classifier accuracy.

- **Fedele et al. (2024)** — Work on **window-occlusion** and **perturbation-based analyses** of **feature contributions** and **failure modes** in Siamese / metric-learning setups (occlusion and local masking as tools to probe *what* the network compares). We use **random rectangular erasing** as a practical analogue of **partial occlusion** on obstacle patches.

### Class definitions (YOLO)

| ID | Class |
|---:|---|
| 0 | potholes |
| 1 | thelas |
| 2 | animals |
| 3 | barricades |
| 4 | rickshaws |


In [ ]:
# --- Imports, device, reproducibility, Colab Drive ---
import os
import random
import math
from pathlib import Path
from collections import defaultdict
from typing import Callable, Dict, List, Optional, Tuple, Any

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import roc_curve, auc

# -----------------------------------------------------------------------------
# USER: set your YOLO dataset root (images + matching .txt labels)
# Expected layouts (any one works if paths match):
#   A) ROOT/images/*.jpg  and  ROOT/labels/*.txt  (same basename)
#   B) ROOT/images/train/*  and  ROOT/labels/train/*
# -----------------------------------------------------------------------------
DATASET_ROOT = "/content/drive/MyDrive/your_dataset_root"  # <-- change this

try:
    get_ipython  # type: ignore
    _in_ipython = True
except NameError:
    _in_ipython = False

IN_COLAB = False
if _in_ipython:
    IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as e:
        print("Colab drive mount skipped or failed:", e)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110


## Section 2: Data Processing Pipeline

We scan full-scene images, read YOLO labels (`class_id cx cy w h` normalized), convert boxes to pixel crops, and **sample same-class vs. different-class patch pairs** for the sensitivity loop. Patches are resized to **224×224** and normalized with **ImageNet** statistics for ResNet.


In [ ]:
CLASS_NAMES = {0: "potholes", 1: "thelas", 2: "animals", 3: "barricades", 4: "rickshaws"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def yolo_norm_to_xyxy(
    cx: float, cy: float, w: float, h: float, W: int, H: int
) -> Tuple[int, int, int, int]:
    x_c, y_c = cx * W, cy * H
    bw, bh = w * W, h * H
    x1 = int(max(0, math.floor(x_c - bw / 2)))
    y1 = int(max(0, math.floor(y_c - bh / 2)))
    x2 = int(min(W, math.ceil(x_c + bw / 2)))
    y2 = int(min(H, math.ceil(y_c + bh / 2)))
    if x2 <= x1:
        x2 = min(W, x1 + 1)
    if y2 <= y1:
        y2 = min(H, y1 + 1)
    return x1, y1, x2, y2


def discover_image_label_pairs(root: Path) -> List[Tuple[Path, Path]]:
    pairs = []
    root = root.expanduser().resolve()
    if not root.is_dir():
        print(f"Warning: DATASET_ROOT does not exist: {root}")
        return pairs

    cand_imgs: List[Path] = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            cand_imgs.append(p)

    for img_path in cand_imgs:
        rel = img_path.relative_to(root)
        parts = list(rel.parts)
        if parts[0].lower() == "images":
            parts[0] = "labels"
        else:
            parts = ["labels"] + parts
        lbl_guess = root.joinpath(*parts).with_suffix(".txt")
        if lbl_guess.is_file():
            pairs.append((img_path, lbl_guess))
            continue
        flat = root / "labels" / f"{img_path.stem}.txt"
        if flat.is_file():
            pairs.append((img_path, flat))
            continue
        same = img_path.with_suffix(".txt")
        if same.is_file():
            pairs.append((img_path, same))

    return pairs


class YOLOObstaclePatchIndex(Dataset):
    # Index: one row per YOLO box (obstacle instance).

    def __init__(self, dataset_root: str, min_side_px: int = 8):
        self.root = Path(dataset_root)
        self.min_side = min_side_px
        self.entries: List[Tuple[Path, int, Tuple[float, float, float, float]]] = []
        pairs = discover_image_label_pairs(self.root)
        for img_path, lbl_path in pairs:
            try:
                with Image.open(img_path) as im:
                    W, H = im.size
            except Exception:
                continue
            try:
                lines = [ln.strip() for ln in lbl_path.read_text().splitlines() if ln.strip()]
            except Exception:
                continue
            for ln in lines:
                parts = ln.split()
                if len(parts) < 5:
                    continue
                c = int(float(parts[0]))
                cx, cy, w, h = map(float, parts[1:5])
                if c not in CLASS_NAMES:
                    continue
                x1, y1, x2, y2 = yolo_norm_to_xyxy(cx, cy, w, h, W, H)
                if (x2 - x1) < self.min_side or (y2 - y1) < self.min_side:
                    continue
                self.entries.append((img_path, c, (cx, cy, w, h)))

        by_c = defaultdict(list)
        for i, (_, c, _) in enumerate(self.entries):
            by_c[c].append(i)
        self.by_class = dict(by_c)
        print(f"Indexed {len(self.entries)} object instances across {len(self.by_class)} classes.")

    def __len__(self) -> int:
        return len(self.entries)

    def crop_pil(self, idx: int) -> Image.Image:
        img_path, _, box = self.entries[idx]
        cx, cy, w, h = box
        with Image.open(img_path).convert("RGB") as im:
            W, H = im.size
            x1, y1, x2, y2 = yolo_norm_to_xyxy(cx, cy, w, h, W, H)
            return im.crop((x1, y1, x2, y2))

    def label_of(self, idx: int) -> int:
        return self.entries[idx][1]

    def sample_same_class_pair_indices(self) -> Tuple[int, int]:
        c = random.choice(list(self.by_class.keys()))
        ix = self.by_class[c]
        if len(ix) < 2:
            return ix[0], ix[0]
        a, b = random.sample(ix, 2)
        return a, b

    def sample_diff_class_pair_indices(self) -> Tuple[int, int]:
        classes = list(self.by_class.keys())
        if len(classes) < 2:
            c1 = c2 = classes[0]
        else:
            c1, c2 = random.sample(classes, 2)
        a = random.choice(self.by_class[c1])
        b = random.choice(self.by_class[c2])
        return a, b


patch_index = YOLOObstaclePatchIndex(DATASET_ROOT)
assert len(patch_index) > 0, (
    "No YOLO boxes indexed — set DATASET_ROOT to a folder with images and matching .txt labels."
)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def tensorize_normalize(pil: Image.Image, size: int = 224) -> torch.Tensor:
    pil = pil.resize((size, size), Image.BICUBIC)
    t = TF.to_tensor(pil)
    t = TF.normalize(t, IMAGENET_MEAN, IMAGENET_STD)
    return t


## Section 3: Model Definition & Perturbation Transforms

We load **ResNet-50** from `torchvision` with pretrained weights, **freeze** all parameters, and **remove the classification head** so the network outputs a **2048-D** embedding after global average pooling.

Perturbations (applied to the **patch** before resize/normalize where noted):

1. **Rotation** — degrees in $\{15,30,45,60,75,90\}$.  
2. **Gaussian blur** — odd kernel sizes and increasing $\sigma$.  
3. **Occlusion / random erasing** — rectangular mask on the **[0,1]** tensor before ImageNet normalization; severity via **fraction of area** erased.


In [ ]:
def build_frozen_resnet50_backbone(device: torch.device) -> nn.Module:
    weights = ResNet50_Weights.IMAGENET1K_V1
    m = resnet50(weights=weights)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Identity()
    m.eval()
    return m.to(device)


backbone = build_frozen_resnet50_backbone(DEVICE)


@torch.no_grad()
def embed_batch(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    # x: Nx3x224x224, ImageNet-normalized for ResNet.
    z = model(x)
    return F.normalize(z, dim=1)


def rotate_pil(pil: Image.Image, degrees: float) -> Image.Image:
    return pil.rotate(degrees, resample=Image.BILINEAR, expand=False)


def random_rect_occlusion_tensor(t01: torch.Tensor, area_frac: float, rng: random.Random) -> torch.Tensor:
    # t01: CxHxW in [0,1]; zero-fill a random rectangle ~area_frac of area.
    c, h, w = t01.shape
    area = h * w
    target = int(area * area_frac)
    if target < 1:
        return t01
    for _ in range(8):
        er_h = rng.randint(1, h)
        er_w = rng.randint(1, w)
        if er_h * er_w < max(1, target // 2):
            continue
        i = rng.randint(0, h - er_h)
        j = rng.randint(0, w - er_w)
        t2 = t01.clone()
        t2[:, i : i + er_h, j : j + er_w] = 0.0
        return t2
    return t01


def pil_to_tensor01_resized(pil: Image.Image, size: int) -> torch.Tensor:
    pil = pil.resize((size, size), Image.BICUBIC)
    return TF.to_tensor(pil)


def normalize_01_tensor(t: torch.Tensor) -> torch.Tensor:
    return TF.normalize(t, IMAGENET_MEAN, IMAGENET_STD)


PERTURBATION_LEVELS: Dict[str, List[Any]] = {
    "rotation": [15, 30, 45, 60, 75, 90],
    "blur": [(3, 0.4), (5, 0.8), (5, 1.6), (7, 2.0), (9, 3.0), (11, 4.0)],
    "occlusion": [0.05, 0.10, 0.20, 0.30, 0.40, 0.50],
}


# torchvision.transforms organized by family (rotation uses fixed angles in PERTURBATION_LEVELS).
PERTURBATION_TRANSFORMS: Dict[str, Any] = {
    "gaussian_blur": [
        T.GaussianBlur(
            kernel_size=(k if k % 2 == 1 else k + 1),
            sigma=(float(s), float(s)),
        )
        for k, s in PERTURBATION_LEVELS["blur"]
    ],
    "random_erasing": [
        T.RandomErasing(
            p=1.0,
            scale=(max(f - 0.02, 0.02), min(f + 0.08, 0.6)),
            ratio=(0.3, 3.3),
            value=0.0,
        )
        for f in PERTURBATION_LEVELS["occlusion"]
    ],
}


def gaussian_blur_pil_at_level(pil: Image.Image, blur_level_idx: int) -> Image.Image:
    t = TF.to_tensor(pil)
    out = PERTURBATION_TRANSFORMS["gaussian_blur"][blur_level_idx](t)
    return TF.to_pil_image(out.clamp(0, 1))


def apply_rotation_clean_pair(pil_a: Image.Image, pil_b: Image.Image, deg: float) -> Tuple[torch.Tensor, torch.Tensor]:
    # A clean; B rotated.
    tb = rotate_pil(pil_b, deg)
    return tensorize_normalize(pil_a), tensorize_normalize(tb)


def apply_blur_on_b(
    pil_a: Image.Image, pil_b: Image.Image, blur_level_idx: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    tb = gaussian_blur_pil_at_level(pil_b, blur_level_idx)
    return tensorize_normalize(pil_a), tensorize_normalize(tb)


def apply_occlusion_on_b(
    pil_a: Image.Image, pil_b: Image.Image, area_frac: float, seed: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    rng = random.Random(seed)
    t01 = pil_to_tensor01_resized(pil_b, 224)
    t01 = random_rect_occlusion_tensor(t01, area_frac, rng)
    emb = normalize_01_tensor(t01)
    return tensorize_normalize(pil_a), emb


print("Perturbation level grids:", {k: len(v) for k, v in PERTURBATION_LEVELS.items()})


## Section 4: Experimental Loop

For each perturbation family and intensity:

- **Same-class pairs:** two independent crops of the same class; **Image A** is **clean** from the first crop; **Image B** is the **perturbed** second crop.  
- **Different-class pairs:** the same protocol on **cross-class** crops.

We record **cosine similarity** (embeddings are **L2-normalized**, so cosine equals dot product) and **$L_2$** in embedding space. **False acceptance rate (FAR)** for different-class pairs uses a threshold set to the **median** of **clean** same-class similarities: FAR = fraction of different-class **perturbed** pairs with similarity $\geq$ threshold.

**ROC:** positives = same-class, negatives = different-class, score = cosine. Section 5 compares **clean** vs. **max-severity** perturbations per family.


In [ ]:
def run_level(
    perturb_name: str,
    level_idx: int,
    n_same: int = 200,
    n_diff: int = 200,
) -> Dict[str, Any]:
    clean_same_scores: List[float] = []
    pert_same_cos: List[float] = []
    pert_same_l2: List[float] = []
    clean_diff_scores: List[float] = []
    pert_diff_cos: List[float] = []
    pert_diff_l2: List[float] = []

    for _ in range(n_same):
        i, j = patch_index.sample_same_class_pair_indices()
        pil_a = patch_index.crop_pil(i)
        pil_b = patch_index.crop_pil(j)
        xa_c = tensorize_normalize(pil_a)
        xb_c = tensorize_normalize(pil_b)
        with torch.no_grad():
            za = embed_batch(backbone, xa_c.unsqueeze(0).to(DEVICE))
            zb0 = embed_batch(backbone, xb_c.unsqueeze(0).to(DEVICE))
        clean_same_scores.append(float((za * zb0).sum().item()))

        if perturb_name == "rotation":
            deg = PERTURBATION_LEVELS["rotation"][level_idx]
            xa, xb = apply_rotation_clean_pair(pil_a, pil_b, deg)
        elif perturb_name == "blur":
            xa, xb = apply_blur_on_b(pil_a, pil_b, level_idx)
        elif perturb_name == "occlusion":
            frac = PERTURBATION_LEVELS["occlusion"][level_idx]
            xa, xb = apply_occlusion_on_b(pil_a, pil_b, frac, seed=random.randint(0, 10_000_000))
        else:
            raise ValueError(perturb_name)

        with torch.no_grad():
            za = embed_batch(backbone, xa.unsqueeze(0).to(DEVICE))
            zb = embed_batch(backbone, xb.unsqueeze(0).to(DEVICE))
        cos = float((za * zb).sum().item())
        l2 = float(torch.norm(za - zb, p=2).item())
        pert_same_cos.append(cos)
        pert_same_l2.append(l2)

    for _ in range(n_diff):
        i, j = patch_index.sample_diff_class_pair_indices()
        pil_a = patch_index.crop_pil(i)
        pil_b = patch_index.crop_pil(j)
        xa_c = tensorize_normalize(pil_a)
        xb_c = tensorize_normalize(pil_b)
        with torch.no_grad():
            za = embed_batch(backbone, xa_c.unsqueeze(0).to(DEVICE))
            zb0 = embed_batch(backbone, xb_c.unsqueeze(0).to(DEVICE))
        clean_diff_scores.append(float((za * zb0).sum().item()))

        if perturb_name == "rotation":
            deg = PERTURBATION_LEVELS["rotation"][level_idx]
            xa, xb = apply_rotation_clean_pair(pil_a, pil_b, deg)
        elif perturb_name == "blur":
            xa, xb = apply_blur_on_b(pil_a, pil_b, level_idx)
        elif perturb_name == "occlusion":
            frac = PERTURBATION_LEVELS["occlusion"][level_idx]
            xa, xb = apply_occlusion_on_b(pil_a, pil_b, frac, seed=random.randint(0, 10_000_000))
        with torch.no_grad():
            za = embed_batch(backbone, xa.unsqueeze(0).to(DEVICE))
            zb = embed_batch(backbone, xb.unsqueeze(0).to(DEVICE))
        pert_diff_cos.append(float((za * zb).sum().item()))
        pert_diff_l2.append(float(torch.norm(za - zb, p=2).item()))

    thr = float(np.median(clean_same_scores))
    far = float(np.mean([s >= thr for s in pert_diff_cos]))

    return {
        "clean_same_mean": float(np.mean(clean_same_scores)),
        "pert_same_cos_mean": float(np.mean(pert_same_cos)),
        "pert_same_l2_mean": float(np.mean(pert_same_l2)),
        "clean_diff_mean": float(np.mean(clean_diff_scores)),
        "pert_diff_cos_mean": float(np.mean(pert_diff_cos)),
        "threshold_median_clean_same": thr,
        "far_at_threshold": far,
        "clean_same_scores": clean_same_scores,
        "pert_same_cos": pert_same_cos,
        "clean_diff_scores": clean_diff_scores,
        "pert_diff_cos": pert_diff_cos,
    }


results_store: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
pert_names = ["rotation", "blur", "occlusion"]

for pname in pert_names:
    nlev = len(PERTURBATION_LEVELS[pname])
    for li in range(nlev):
        print(f"Running {pname} level {li + 1}/{nlev} ...")
        results_store[pname].append(run_level(pname, li, n_same=150, n_diff=150))

print("Done collecting curves.")


## Section 5: Results & Visualizations

1. **Perturbation robustness curves** — mean cosine vs. intensity (x-axis shows human-readable severity).  
2. **ROC curves** — clean baseline vs. **highest** intensity per perturbation family.  
3. **Failure taxonomy grid** — example pairs: **same-class** similarity collapse under strong blur; **different-class** false high similarity under clean viewing (heuristic threshold).


In [ ]:
def xtick_labels_for(pname: str) -> List[str]:
    if pname == "rotation":
        return [str(d) + "°" for d in PERTURBATION_LEVELS["rotation"]]
    if pname == "blur":
        return [f"k{k},σ{s}" for k, s in PERTURBATION_LEVELS["blur"]]
    return [f"{int(f * 100)}%" for f in PERTURBATION_LEVELS["occlusion"]]


def plot_robustness_curves(results_store: Dict[str, List[dict]]):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="col")
    for col, pname in enumerate(["rotation", "blur", "occlusion"]):
        rows = results_store[pname]
        xs = np.arange(len(rows))
        ys_same = [r["pert_same_cos_mean"] for r in rows]
        ys_diff = [r["pert_diff_cos_mean"] for r in rows]
        l2_same = [r["pert_same_l2_mean"] for r in rows]
        far = [r["far_at_threshold"] for r in rows]
        ax = axes[0, col]
        ax.plot(xs, ys_same, "o-", label="Same-class cos")
        ax.plot(xs, ys_diff, "s-", label="Diff-class cos")
        ax.set_xticks(xs)
        ax.set_xticklabels(xtick_labels_for(pname), rotation=35, ha="right")
        ax.set_ylabel("Mean cosine")
        ax.set_title(pname.capitalize())
        ax.legend(fontsize=7)
        ax2 = ax.twinx()
        ax2.plot(xs, l2_same, "^--", color="tab:gray", alpha=0.8, label="Same-class L2")
        ax2.set_ylabel("Mean L2 (embed)")
        ax2.legend(fontsize=7, loc="lower right")

        axb = axes[1, col]
        axb.bar(xs, far, color="tab:red", alpha=0.7)
        axb.set_xticks(xs)
        axb.set_xticklabels(xtick_labels_for(pname), rotation=35, ha="right")
        axb.set_ylabel("FAR (diff-class)")
        axb.set_ylim(0, max(0.05, max(far) * 1.15) if far else (0, 1))
    fig.suptitle("Robustness: cosine / L2 (top), false acceptance rate (bottom)")
    plt.tight_layout()
    plt.show()


plot_robustness_curves(results_store)


def collect_roc_scores(
    mode: str, pname: str, level_idx: int, n_pos: int = 400, n_neg: int = 400
) -> Tuple[np.ndarray, np.ndarray]:
    # mode 'clean': both crops clean; 'perturb': perturb B per pname/level_idx.
    y: List[int] = []
    s: List[float] = []
    for _ in range(n_pos):
        i, j = patch_index.sample_same_class_pair_indices()
        pil_a = patch_index.crop_pil(i)
        pil_b = patch_index.crop_pil(j)
        if mode == "clean":
            xa, xb = tensorize_normalize(pil_a), tensorize_normalize(pil_b)
        elif pname == "rotation":
            deg = PERTURBATION_LEVELS["rotation"][level_idx]
            xa, xb = apply_rotation_clean_pair(pil_a, pil_b, deg)
        elif pname == "blur":
            xa, xb = apply_blur_on_b(pil_a, pil_b, level_idx)
        elif pname == "occlusion":
            frac = PERTURBATION_LEVELS["occlusion"][level_idx]
            xa, xb = apply_occlusion_on_b(pil_a, pil_b, frac, seed=random.randint(0, 10_000_000))
        else:
            raise ValueError(pname)
        with torch.no_grad():
            za = embed_batch(backbone, xa.unsqueeze(0).to(DEVICE))
            zb = embed_batch(backbone, xb.unsqueeze(0).to(DEVICE))
        s.append(float((za * zb).sum().item()))
        y.append(1)
    for _ in range(n_neg):
        i, j = patch_index.sample_diff_class_pair_indices()
        pil_a = patch_index.crop_pil(i)
        pil_b = patch_index.crop_pil(j)
        if mode == "clean":
            xa, xb = tensorize_normalize(pil_a), tensorize_normalize(pil_b)
        elif pname == "rotation":
            deg = PERTURBATION_LEVELS["rotation"][level_idx]
            xa, xb = apply_rotation_clean_pair(pil_a, pil_b, deg)
        elif pname == "blur":
            xa, xb = apply_blur_on_b(pil_a, pil_b, level_idx)
        elif pname == "occlusion":
            frac = PERTURBATION_LEVELS["occlusion"][level_idx]
            xa, xb = apply_occlusion_on_b(pil_a, pil_b, frac, seed=random.randint(0, 10_000_000))
        with torch.no_grad():
            za = embed_batch(backbone, xa.unsqueeze(0).to(DEVICE))
            zb = embed_batch(backbone, xb.unsqueeze(0).to(DEVICE))
        s.append(float((za * zb).sum().item()))
        y.append(0)
    return np.array(y), np.array(s)


def plot_roc_comparison():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, key in zip(axes, ["rotation", "blur", "occlusion"]):
        last = len(PERTURBATION_LEVELS[key]) - 1
        y0, s0 = collect_roc_scores("clean", key, 0, n_pos=250, n_neg=250)
        y1, s1 = collect_roc_scores("perturb", key, last, n_pos=250, n_neg=250)
        fpr0, tpr0, _ = roc_curve(y0, s0)
        fpr1, tpr1, _ = roc_curve(y1, s1)
        ax.plot(fpr0, tpr0, label=f"Clean (AUC={auc(fpr0, tpr0):.3f})")
        ax.plot(fpr1, tpr1, label=f"Max {key} (AUC={auc(fpr1, tpr1):.3f})")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.set_title(f"ROC: clean vs max {key}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_roc_comparison()


@torch.no_grad()
def score_pair_tensors(xa: torch.Tensor, xb: torch.Tensor) -> float:
    za = embed_batch(backbone, xa.unsqueeze(0).to(DEVICE))
    zb = embed_batch(backbone, xb.unsqueeze(0).to(DEVICE))
    return float((za * zb).sum().item())


def failure_taxonomy_grid(max_examples: int = 4):
    # Heuristic threshold from clean same-class score distribution.
    ref_scores = []
    for _ in range(120):
        i, j = patch_index.sample_same_class_pair_indices()
        xa = tensorize_normalize(patch_index.crop_pil(i))
        xb = tensorize_normalize(patch_index.crop_pil(j))
        ref_scores.append(score_pair_tensors(xa, xb))
    thr = float(np.percentile(ref_scores, 25))

    fails_same: List[Tuple[str, Image.Image, Image.Image, float]] = []
    fails_diff: List[Tuple[str, Image.Image, Image.Image, float]] = []

    bi = len(PERTURBATION_LEVELS["blur"]) - 1

    attempts = 0
    while (len(fails_same) < max_examples or len(fails_diff) < max_examples) and attempts < 800:
        attempts += 1
        if len(fails_same) < max_examples:
            i, j = patch_index.sample_same_class_pair_indices()
            pa, pb = patch_index.crop_pil(i), patch_index.crop_pil(j)
            xa, xb = apply_blur_on_b(pa, pb, bi)
            sc = score_pair_tensors(xa, xb)
            if sc < thr:
                fails_same.append(("same-class | strong blur", pa, pb, sc))
        if len(fails_diff) < max_examples:
            i, j = patch_index.sample_diff_class_pair_indices()
            pa, pb = patch_index.crop_pil(i), patch_index.crop_pil(j)
            xa, xb = tensorize_normalize(pa), tensorize_normalize(pb)
            sc = score_pair_tensors(xa, xb)
            if sc >= thr:
                fails_diff.append(("diff-class | false high sim (clean)", pa, pb, sc))

    n = max(len(fails_same), len(fails_diff), 1)
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 5))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])
    for col in range(n):
        if col < len(fails_same):
            title, pa, pb, sc = fails_same[col]
            axes[0, col].imshow(pa.resize((224, 224)))
            axes[0, col].set_title(f"{title}\ncos={sc:.3f}", fontsize=8)
        else:
            axes[0, col].axis("off")
        if col < len(fails_diff):
            title, pa, pb, sc = fails_diff[col]
            axes[1, col].imshow(
                np.hstack([np.array(pa.resize((112, 112))), np.array(pb.resize((112, 112)))])
            )
            axes[1, col].set_title(f"{title}\ncos={sc:.3f}", fontsize=8)
        else:
            axes[1, col].axis("off")
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    plt.suptitle("Failure taxonomy (heuristic): top=same-class collapse; bottom=false accept", y=1.02)
    plt.tight_layout()
    plt.show()


failure_taxonomy_grid(max_examples=4)
